# SC3000 DPO — reproducible final E5 training notebook

**Final selected model:** Round9 Random E5 (historical development Val768 614/768, Teacher680 521/680). This is a multi-stage curriculum, **not** five epochs directly from SFT.

This notebook provides the **executable training loop**, explicit data preparation, hash-based provenance, epoch-boundary resume, identical course loss and greedy inference, and a final dpo/dpo.pt export. Open from repository root or dpo/, set FAST_SMOKE=False, and choose **Kernel → Restart & Run All**.

**Historical chain (all stages start fresh AdamW; see stage-specific cosine schedules below):**

1. SFT → Round3 full 100k structured, five epochs, LR 1e-4.
2. Round3 → Round4 Treatment **epoch 1 of a two-epoch cosine schedule**, LR 2e-5.
3. Round4 E1 → Round5 Original (one-step because), one epoch, LR 1e-5.
4. Round5 Original → Round7 Replay, one epoch, LR 1e-5.
5. Round7 Replay → Round9 Random E1, one epoch, LR 1e-5.
6. Random E1 → Random E2, one epoch, LR 1e-5, **fresh optimizer** (original E1 did not store optimizer state).
7. Random E2 → Random E3, E4, **E5** (three actual epochs on a **four-epoch cosine schedule**), LR 1e-5, one continuous optimizer across E3–E5.

**Required starter files:** supplied sft/gpt.pt, sft/meta.pkl, the course model.py, and source scripts from this GitHub repository. The notebook automatically builds missing source datasets using the audited generator scripts (their SHA256 is included in its report). Do **not** upload only dpo.pt and claim reproduction from scratch. Preserve all generated training JSON and the generator scripts with the assignment package; the historical project datasets are large and not tracked by Git.

**Integrity:** Each dataset and parent checkpoint is hashed, factual answers/format are checked, stage checkpoints store optimizer, scheduler, Python/Torch RNG and shuffled training order. If interrupted, rerun **Run All**; completed stages are verified and skipped. Historical checkpoints elsewhere in the repo are never overwritten. Round3 final Test768 is **never evaluated** for model selection; frozen Val768 and Seen-heavy Teacher680 are development benchmarks.

**Limits:** Exact numerical identity with the archived historical E5 cannot be guaranteed across GPUs, PyTorch releases, nondeterministic CUDA operations or different source code commits. This notebook reports a comparison when the archived E5 is available rather than falsely promising byte-level parity. Historical archived scores are **reference results**, not a claim that the new run has already been executed. The TC1 Slurm 6-hour per-job QoS may require **rerunning Run All to resume**; a single 6-hour allocation cannot be guaranteed to complete the full historical chain.

In [ ]:
# STEP 1 — Imports, deterministic protocol and runtime paths
from __future__ import annotations
import os, sys, json, pickle, random, math, hashlib, shutil, time, re, subprocess
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone

import torch
import torch.nn.functional as F

workdir = Path.cwd().resolve()
if (workdir / "model.py").is_file() and (workdir / "dpo" / "dpo.ipynb").exists():
    repo_root = workdir
elif (workdir / "dpo.ipynb").is_file() and (workdir.parent / "model.py").is_file():
    repo_root = workdir.parent
else:
    raise RuntimeError("Open from NanoGPT-Math root/ or NanoGPT-Math/dpo/.")

sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(repo_root / "scripts"))
from model import GPT, GPTConfig

# Keep the originally supplied pretrained 8.84M-parameter NanoGPT checkpoint.
SFT = repo_root / "sft" / "gpt.pt"
META = repo_root / "sft" / "meta.pkl"
for required in (SFT, META):
    if not required.is_file():
        raise FileNotFoundError(f"Missing REQUIRED course-provided input: {required}")

with META.open("rb") as fh:
    vocab = pickle.load(fh)
stoi, itos = vocab["stoi"], vocab["itos"]
assert len(stoi) == len(itos) == 74, "Expected original 74-character vocabulary"
assert stoi["\n"] == 0 and stoi["."] == 7, "Original stopping IDs changed"

# FULL: 13 historical 100k training epochs across seven stages.
# SMOKE: replace each 100k stage by a 128-row mini-stage and NEVER publish dpo.pt.
FAST_SMOKE = False
AUTO_BUILD_MISSING_DATA = True
MAX_NEW_TOKENS = 80
BATCH_SIZE = 64
SEED = 42
BETA = 0.5
TRAIN_BASE_LENGTH = 64
device = "cuda" if torch.cuda.is_available() else "cpu"
RUN_DIR = (repo_root / "dpo" / "experiments" /
           ("final_notebook_smoke" if FAST_SMOKE else "final_notebook_e5_reproduction"))
RUN_DIR.mkdir(parents=True, exist_ok=True)

def sha256(path):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as fh:
        for part in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(part)
    return h.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def save_json_once(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        raise FileExistsError(f"Will not overwrite frozen artifact: {path}")
    temporary = path.with_suffix(path.suffix + ".partial")
    if temporary.exists():
        raise RuntimeError(f"Stale incomplete artifact, inspect first: {temporary}")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\n",
                         encoding="utf-8")
    temporary.rename(path)

def record_environment():
    try:
        git_rev = subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo_root,
                                text=True, check=True, capture_output=True).stdout.strip()
    except (OSError, subprocess.CalledProcessError):
        git_rev = "unknown (archive export)"
    return {"utc":datetime.now(timezone.utc).isoformat(),
            "python":sys.version, "torch":torch.__version__,
            "cuda_runtime":torch.version.cuda,
            "gpu":torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "git_revision":git_rev,
            "sft_sha256":sha256(SFT), "tokenizer_meta_sha256":sha256(META)}

env_record=record_environment()
print(json.dumps(env_record, indent=2), flush=True)
if FAST_SMOKE:
    print("SMOKE mode: never export its model as formal dpo.pt")
elif device != "cuda":
    raise RuntimeError("Full 13-epoch reproduction requires CUDA. "
                       "Use a Slurm GPU/Jupyter allocation or set FAST_SMOKE=True.")

## Task 1 — Preference data construction and isolation

The data builders are the original audited project implementation. This cell calls them **automatically**, in source-dependency order, if their immutable manifest is absent; it never silently regenerates a frozen dataset. Each builder verifies exact arithmetic, sentence format, 74-character vocabulary, 256-character NanoGPT block limit, and canonical-math-fact splits (seed 42).

Files in the historical path:

- Round1A → Round1B integer-only → Round1C historical fixed OOD256 (used only to isolate later generated facts; **not** an optimization objective here).
- Round3: 100k preference pairs; validation 768; **final test is sealed**.
- Round4 Treatment: 70k common replay + 15k hard seen training + 15k new train facts; extended validation kept out of training.
- Round5 Original: Round4 Treatment questions and one-step explanations preserved; compare with the alternative decomposition branch separately in the research log.
- Round7 Replay: controlled same-size 100k replay.
- Round9 Random: 70k common old + 30k additional training-partition questions, frozen after Round7 checkpoint exists.

Training format: positive = question + ' The answer is [integer] because [verified equality].'; negative = same question + " Sorry, I don't know.". We keep historical repeated exposures and log unique prompt/fact counts. Changing them here would constitute a new experiment, not the same E5.

In [ ]:
# STEP 2 — Generate *missing* immutable data automatically (no manual shell steps).
# All builder programs live in this same GitHub repository, and their
# exact source SHA256 is stored in the final run provenance.
GENERATOR_PLAN = [
    ("Round1A frozen source", repo_root/"dpo/data/round1_full/round1_manifest.json",
     "scripts/generate_round1.py", ["--profile", "full"]),
    ("Round1B integer source", repo_root/"dpo/experiments/round1b_integer/round1b_manifest.json",
     "scripts/generate_round1b.py", []),
    ("Round1C fixed validation", repo_root/"dpo/experiments/round1c_incorrect/round1c_manifest.json",
     "scripts/generate_round1c.py", []),
    ("Round3 structured 100k", repo_root/"dpo/data/round3_structured_full/manifest.json",
     "scripts/generate_round3_structured.py", ["--profile", "full"]),
    ("Round4 matched data", repo_root/"dpo/data/round4_matched_full/manifest.json",
     "scripts/build_round4_data.py", ["--profile", "full"]),
    ("Round5 one-step data", repo_root/"dpo/data/round5_decomposition/manifest.json",
     "scripts/build_round5_decomposition.py", []),
    ("Round7 paired replay", repo_root/"dpo/data/round7_targeted_full/manifest.json",
     "scripts/build_round7_targeted_data.py", ["--profile", "full"]),
]
generator_provenance = {}

for name, manifest_path, script_rel, args in GENERATOR_PLAN:
    script = repo_root / script_rel
    if not script.is_file():
        raise FileNotFoundError(f"Reproducibility requires source generator {script_rel}")
    generator_provenance[script_rel] = sha256(script)
    if not manifest_path.exists():
        if not AUTO_BUILD_MISSING_DATA:
            raise FileNotFoundError(f"{name}: missing frozen source {manifest_path}")
        print(f"GENERATE {name}: {script_rel} {args}", flush=True)
        subprocess.run([sys.executable, str(script), *args],
                       cwd=repo_root, check=True)
    if not manifest_path.is_file():
        raise RuntimeError(f"{name} generation did not produce {manifest_path}")
    generator_provenance[str(manifest_path.relative_to(repo_root))] = sha256(manifest_path)
    print(f"PREPARED {name}: {manifest_path.relative_to(repo_root)}", flush=True)

# Frozen teacher-style 680 is for retention diagnostics only; its
# arithmetic facts were historically seen, never to be called unseen OOD.
TEACHER_DIRS = [
    (repo_root/"dpo/benchmarks/teacher_style_400_v1", 400),
    (repo_root/"dpo/benchmarks/teacher_style_extension_280_v1", 280),
]
for directory, expected_n in TEACHER_DIRS:
    questions = read_json(directory/"questions.json")
    manifest = read_json(directory/"manifest.json")
    assert len(questions) == expected_n
    assert sha256(directory/"questions.json") == manifest["questions_sha256"]
    generator_provenance[str((directory/"questions.json").relative_to(repo_root))] = sha256(directory/"questions.json")
print("PASS: 400 + 280 frozen teacher questions", flush=True)

## Task 2 — Model and original course preference objective

We intentionally preserve the instructor's original **whole-sequence mean log-probability** loss (not standard frozen-reference DPO). The positive/negative are teacher-forced character sequences: the GPT does **not** generate either answer during training.

For each pair, sequence log-probability is the negative average token cross entropy over non-padding tokens, including the math question. With instructor beta = 0.5:

Loss = mean[softplus(-(logp_chosen - logp_rejected)/beta)] - 0.1 * mean(logp_chosen).

The final term encourages the correct answer sequence itself. All training stages use the original GPTConfig, 74-character tokenizer, block_size=256, **same four appended newline characters**, and max_length adjusted upward if necessary to avoid truncating valid examples.

In [ ]:
# STEP 3 — Course-faithful teacher-forcing loss and preprocessing.
from generate_round1 import parse_relation, stable_split

def encode(s):
    return [stoi[ch] for ch in s]

def pad_or_truncate(chars, max_length):
    return chars[-max_length:] if len(chars)>max_length else chars+[0]*(max_length-len(chars))

def compute_logprob(gpt, batch_ids):
    inputs = batch_ids[:, :-1]
    targets = batch_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    loss = F.cross_entropy(logits.reshape(-1,logits.size(-1)),
                           targets.reshape(-1), ignore_index=0,
                           reduction="none").reshape_as(targets)
    mask = (targets!=0).float()
    return -(loss*mask).sum(dim=1)/mask.sum(dim=1)

def course_preference_loss(gpt, negatives, positives):
    neg_lp = compute_logprob(gpt, negatives)
    pos_lp = compute_logprob(gpt, positives)
    return -F.logsigmoid((pos_lp-neg_lp)/BETA).mean()-0.1*pos_lp.mean()

def get_batches(lines, batch_size, max_length, target_device):
    # EXACT instructor rule: shuffle rows in place, discard trailing <batch.
    random.shuffle(lines)
    for i in range(0,len(lines),batch_size):
        batch=lines[i:i+batch_size]
        if len(batch)<batch_size:continue
        neg=[pad_or_truncate(encode(x["negative"]+"\n\n\n\n"),max_length) for x in batch]
        pos=[pad_or_truncate(encode(x["positive"]+"\n\n\n\n"),max_length) for x in batch]
        yield (torch.tensor(neg,dtype=torch.long,device=target_device),
               torch.tensor(pos,dtype=torch.long,device=target_device))

def verify_pairs(path, expected_count=100000, smoke=False):
    source = read_json(path)
    if len(source)!=expected_count:
        raise RuntimeError(f"{path}: expected {expected_count}, found {len(source)}")
    rows = source[:128] if smoke else source
    max_required = TRAIN_BASE_LENGTH
    unique_prompts=set()
    unique_facts=set()
    for i,row in enumerate(rows):
        pos,neg=row["positive"],row["negative"]
        marker=" The answer is "
        if marker not in pos or " because " not in pos:
            raise ValueError(f"{path.name} row {i}: incorrect chosen format")
        question,answer_suffix=pos.split(marker,1)
        answer=answer_suffix.split(" because ",1)[0]
        if not re.fullmatch(r"[+-]?\d+",answer) or not pos.endswith("."):
            raise ValueError(f"{path.name} row {i}: malformed integer chosen")
        if neg!=question+" Sorry, I don't know.":
            raise ValueError(f"{path.name} row {i}: changed rejection/prompt")
        if set(pos+neg)-set(stoi):
            raise ValueError(f"{path.name} row {i}: tokenizer OOV")
        max_required=max(max_required,len(pos)+4,len(neg)+4)
        unique_prompts.add(question)
        unique_facts.add(parse_relation(question,answer))
    # Original NanoGPT block size=256; 64 was only the initial pack length.
    if max_required>256:
        raise ValueError(f"{path.name}: {max_required} exceeds model block_size")
    info={"source":str(path.relative_to(repo_root)),"sha256":sha256(path),
          "rows":len(rows),"unique_prompts":len(unique_prompts),
          "unique_facts":len(unique_facts),"max_length":max_required}
    return rows,info

def model_from_checkpoint(path):
    obj=torch.load(path,map_location="cpu",weights_only=True)
    args=obj["model_args"]
    model=GPT(GPTConfig(**args))
    state=obj.get("model_state_dict",obj.get("model"))
    if state is None:raise RuntimeError(f"{path}: missing state dict")
    model.load_state_dict({k.removeprefix("_orig_mod."):v for k,v in state.items()},strict=True)
    return model,args

# Sanity checks: answer-first comparison, no generated completion needed in training.
assert encode("3*17=?")
assert abs(float(F.logsigmoid(torch.tensor(0.0)))+math.log(2))<1e-6
print("PASS: course loss, tokenizer, model loading and batch conventions",flush=True)